# K-Means Clustering from Scratch

**What we'll build:** A complete k-means clustering algorithm from first principles

**Why it matters:** K-means is one of the most fundamental unsupervised learning algorithms. Understanding how it works from scratch builds intuition for:
- How algorithms discover patterns without labels
- The iterative nature of optimization
- The role of initialization in algorithm behavior
- Distance-based similarity measures

**What you'll learn:**
- How k-means partitions data into clusters
- Why the algorithm converges (and when it doesn't)
- How initialization affects final results
- Practical considerations for real-world use

## 1. Introduction: What is Clustering?

**Clustering** is the task of grouping similar data points together without any labels. Unlike classification (supervised learning), we don't have target labels — we want the algorithm to discover natural groupings in the data.

**K-means** finds $k$ cluster centers (centroids) and assigns each data point to the nearest centroid. It's simple, fast, and works well when clusters are roughly spherical and similar in size.

## 2. Setup

Let's import the libraries we'll need. We'll use NumPy for computations and Matplotlib for visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from typing import Tuple, List

# Set random seed for reproducibility
np.random.seed(42)

Let's configure our plotting style for cleaner visualizations.

In [ ]:
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 3. Generate Sample Data

Before we build k-means, we need some data to cluster. Let's create synthetic 2D data with clear clusters so we can visualize what's happening.

In [ ]:
def generate_clusters(n_samples: int = 300, n_clusters: int = 3, cluster_std: float = 0.6) -> np.ndarray:
    """
    Generate synthetic 2D data with well-separated Gaussian clusters.
    
    Args:
        n_samples: Total number of points
        n_clusters: Number of clusters
        cluster_std: Standard deviation of clusters
    
    Returns:
        Array of shape (n_samples, 2)
    """
    samples_per_cluster = n_samples // n_clusters
    
    # Generate cluster centers
    centers = np.array([
        [0, 0],
        [4, 4],
        [0, 4]
    ])
    
    # Generate points around each center
    data = []
    for center in centers[:n_clusters]:
        cluster_data = np.random.randn(samples_per_cluster, 2) * cluster_std + center
        data.append(cluster_data)
    
    return np.vstack(data)

# Generate our dataset
X = generate_clusters(n_samples=300, n_clusters=3)
print(f"Generated {len(X)} data points with shape {X.shape}")

Let's visualize our generated data to see what we're working with.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Our Data (No Labels - Unsupervised Learning)')
plt.grid(True, alpha=0.3)
plt.show()

**Key observation:** We can see there are 3 natural groupings, but the algorithm doesn't know this — it will discover them purely from the data's geometry.

## 4. Building Block 1: Distance Metric

### The Foundation: Euclidean Distance

K-means is fundamentally about **distance**. To group similar points, we need to measure how close points are to each other.

The **Euclidean distance** between two points $\mathbf{x}$ and $\mathbf{y}$ in 2D is:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{(x_1 - y_1)^2 + (x_2 - y_2)^2}$$

This is just the straight-line distance you'd measure with a ruler.

In [ ]:
def euclidean_distance(x: np.ndarray, y: np.ndarray) -> float:
    """
    Calculate Euclidean distance between two points.
    
    Args:
        x: First point (1D array)
        y: Second point (1D array)
    
    Returns:
        Distance as a scalar
    """
    return np.sqrt(np.sum((x - y) ** 2))

# Test it
point_a = np.array([0, 0])
point_b = np.array([3, 4])
distance = euclidean_distance(point_a, point_b)
print(f"Distance from {point_a} to {point_b}: {distance:.2f}")
print(f"Expected: 5.00 (classic 3-4-5 triangle)")

**Insight:** This 3-4-5 triangle is a Pythagorean triple — the distance is exactly 5. Our function works correctly!

### Vectorized Distance Computation

For efficiency, we need to compute distances between one point and many points simultaneously. Let's write a vectorized version.

In [ ]:
def compute_distances(X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """
    Compute distances from each point in X to each centroid.
    
    Args:
        X: Data points, shape (n_samples, n_features)
        centroids: Cluster centers, shape (n_clusters, n_features)
    
    Returns:
        Distance matrix, shape (n_samples, n_clusters)
    """
    n_samples = X.shape[0]
    n_clusters = centroids.shape[0]
    distances = np.zeros((n_samples, n_clusters))
    
    for k in range(n_clusters):
        # Distance from all points to centroid k
        distances[:, k] = np.sqrt(np.sum((X - centroids[k]) ** 2, axis=1))
    
    return distances

# Test with a few points and centroids
test_points = np.array([[0, 0], [1, 1], [5, 5]])
test_centroids = np.array([[0, 0], [5, 5]])
distances = compute_distances(test_points, test_centroids)
print("Distance matrix:")
print(distances)
print("\nRows = points, Columns = centroids")

**Key insight:** Point [0,0] is closest to centroid 0, point [5,5] is closest to centroid 1, and point [1,1] is closer to centroid 0. This distance matrix is the foundation of clustering!

## 5. Building Block 2: Initialization

### Choosing Initial Centroids

K-means needs to start somewhere. We need to place $k$ initial centroids in the data space. The simplest approach: **randomly select $k$ points from the data**.

This is called **random initialization**.

In [ ]:
def initialize_centroids(X: np.ndarray, k: int) -> np.ndarray:
    """
    Initialize centroids by randomly selecting k data points.
    
    Args:
        X: Data points, shape (n_samples, n_features)
        k: Number of clusters
    
    Returns:
        Initial centroids, shape (k, n_features)
    """
    indices = np.random.choice(X.shape[0], size=k, replace=False)
    return X[indices].copy()

# Initialize 3 centroids
k = 3
initial_centroids = initialize_centroids(X, k)
print(f"Initial centroids:\n{initial_centroids}")

Let's visualize where these initial centroids landed.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.4, s=50, label='Data points')
plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Initial centroids')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Random Initialization of Centroids')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Key observation:** The initial centroids are random data points. They may not be in ideal positions yet — that's what the algorithm will fix!

## 6. Building Block 3: Assignment Step

### Assigning Points to Nearest Centroid

This is the first of two steps that k-means alternates between:

**Assignment step:** For each data point, find the closest centroid and assign the point to that cluster.

Mathematically, for point $\mathbf{x}_i$, assign it to cluster $c_i$:

$$c_i = \arg\min_k \|\mathbf{x}_i - \mathbf{\mu}_k\|^2$$

where $\mathbf{\mu}_k$ is the $k$-th centroid.

In [ ]:
def assign_clusters(X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """
    Assign each point to the nearest centroid.
    
    Args:
        X: Data points, shape (n_samples, n_features)
        centroids: Current centroids, shape (k, n_features)
    
    Returns:
        Cluster labels, shape (n_samples,)
    """
    distances = compute_distances(X, centroids)
    return np.argmin(distances, axis=1)

# Assign points to initial centroids
labels = assign_clusters(X, initial_centroids)
print(f"Cluster labels shape: {labels.shape}")
print(f"First 10 labels: {labels[:10]}")
print(f"Unique clusters: {np.unique(labels)}")

Let's visualize the initial cluster assignments.

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['blue', 'green', 'orange', 'purple', 'brown']

for i in range(k):
    cluster_points = X[labels == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], 
               alpha=0.6, s=50, c=colors[i], label=f'Cluster {i}')

plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1],
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Centroids')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Initial Cluster Assignments')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Key insight:** Each point is colored by its assigned cluster. Notice that assignments might not be perfect yet — the centroids will move to better positions in the next step.

## 7. Building Block 4: Update Step

### Moving Centroids to Cluster Centers

This is the second step that k-means alternates with assignment:

**Update step:** Move each centroid to the mean (center) of all points assigned to its cluster.

Mathematically, update centroid $\mathbf{\mu}_k$:

$$\mathbf{\mu}_k = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i$$

where $C_k$ is the set of points in cluster $k$.

In [ ]:
def update_centroids(X: np.ndarray, labels: np.ndarray, k: int) -> np.ndarray:
    """
    Update centroids to the mean of their assigned points.
    
    Args:
        X: Data points, shape (n_samples, n_features)
        labels: Current cluster assignments, shape (n_samples,)
        k: Number of clusters
    
    Returns:
        New centroids, shape (k, n_features)
    """
    n_features = X.shape[1]
    centroids = np.zeros((k, n_features))
    
    for i in range(k):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            centroids[i] = cluster_points.mean(axis=0)
        else:
            # If cluster is empty, reinitialize randomly
            centroids[i] = X[np.random.choice(X.shape[0])]
    
    return centroids

# Update centroids based on current assignments
new_centroids = update_centroids(X, labels, k)
print(f"Old centroids:\n{initial_centroids}")
print(f"\nNew centroids:\n{new_centroids}")

Let's visualize how the centroids moved.

In [ ]:
plt.figure(figsize=(8, 6))

# Plot clusters
for i in range(k):
    cluster_points = X[labels == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
               alpha=0.4, s=50, c=colors[i])

# Plot old and new centroids
plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1],
           c='gray', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Old centroids', alpha=0.5)
plt.scatter(new_centroids[:, 0], new_centroids[:, 1],
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='New centroids')

# Draw arrows showing movement
for i in range(k):
    plt.arrow(initial_centroids[i, 0], initial_centroids[i, 1],
             new_centroids[i, 0] - initial_centroids[i, 0],
             new_centroids[i, 1] - initial_centroids[i, 1],
             head_width=0.15, head_length=0.15, fc='red', ec='red', alpha=0.6)

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Centroid Movement After One Update')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Key insight:** The centroids moved toward the center of their assigned clusters. This is why it's called k-**means** — we're computing the mean position!

## 8. Building Block 5: Convergence

### When to Stop?

K-means alternates between assignment and update steps. But when should it stop?

**Convergence** occurs when centroids stop moving (or move very little). We can check:
1. If centroid positions change less than some threshold $\epsilon$
2. If cluster assignments don't change
3. After a maximum number of iterations

We'll check if centroids have moved less than $\epsilon = 0.0001$.

In [ ]:
def has_converged(old_centroids: np.ndarray, new_centroids: np.ndarray, tolerance: float = 1e-4) -> bool:
    """
    Check if centroids have converged.
    
    Args:
        old_centroids: Previous centroids
        new_centroids: Current centroids
        tolerance: Convergence threshold
    
    Returns:
        True if converged, False otherwise
    """
    distances = np.sqrt(np.sum((old_centroids - new_centroids) ** 2, axis=1))
    return np.all(distances < tolerance)

# Test convergence
test_old = np.array([[0, 0], [1, 1]])
test_new_close = np.array([[0.00001, 0.00001], [1.00001, 1.00001]])
test_new_far = np.array([[0.1, 0.1], [1.1, 1.1]])

print(f"Close centroids converged? {has_converged(test_old, test_new_close)}")
print(f"Far centroids converged? {has_converged(test_old, test_new_far)}")

**Key insight:** Convergence means the algorithm has found a stable solution. The clusters won't change anymore, so we can stop.

## 9. Putting It All Together: The Complete Algorithm

### Full K-Means Implementation

Now we combine all the building blocks into the complete k-means algorithm:

1. **Initialize** $k$ centroids randomly
2. **Repeat** until convergence:
   - **Assign** each point to nearest centroid
   - **Update** each centroid to mean of its cluster
3. **Return** final centroids and labels

In [ ]:
def kmeans(X: np.ndarray, k: int, max_iters: int = 100, tolerance: float = 1e-4) -> Tuple[np.ndarray, np.ndarray, List]:
    """
    K-means clustering algorithm.
    
    Args:
        X: Data points, shape (n_samples, n_features)
        k: Number of clusters
        max_iters: Maximum iterations
        tolerance: Convergence threshold
    
    Returns:
        centroids: Final cluster centers
        labels: Final cluster assignments
        history: List of (centroids, labels) at each iteration
    """
    # Initialize
    centroids = initialize_centroids(X, k)
    history = []
    
    for iteration in range(max_iters):
        # Assignment step
        labels = assign_clusters(X, centroids)
        
        # Save for visualization
        history.append((centroids.copy(), labels.copy()))
        
        # Update step
        new_centroids = update_centroids(X, labels, k)
        
        # Check convergence
        if has_converged(centroids, new_centroids, tolerance):
            print(f"Converged after {iteration + 1} iterations")
            centroids = new_centroids
            labels = assign_clusters(X, centroids)
            history.append((centroids.copy(), labels.copy()))
            break
        
        centroids = new_centroids
    else:
        print(f"Reached maximum iterations ({max_iters})")
    
    return centroids, labels, history

# Run k-means!
final_centroids, final_labels, history = kmeans(X, k=3)
print(f"\nFinal centroids:\n{final_centroids}")

Let's visualize the final clustering result.

In [ ]:
plt.figure(figsize=(8, 6))

# Plot final clusters
for i in range(k):
    cluster_points = X[final_labels == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
               alpha=0.6, s=50, c=colors[i], label=f'Cluster {i}')

# Plot final centroids
plt.scatter(final_centroids[:, 0], final_centroids[:, 1],
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Final centroids')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Final K-Means Clustering Result')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Key insight:** The algorithm successfully discovered the three natural clusters! The centroids are now positioned at the center of each group.

### Visualizing the Algorithm's Evolution

Let's see how the clusters evolved over iterations.

In [ ]:
# Plot first few iterations
n_iterations_to_show = min(6, len(history))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(n_iterations_to_show):
    ax = axes[idx]
    centroids, labels = history[idx]
    
    # Plot clusters
    for i in range(k):
        cluster_points = X[labels == i]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1],
                  alpha=0.6, s=30, c=colors[i])
    
    # Plot centroids
    ax.scatter(centroids[:, 0], centroids[:, 1],
              c='red', marker='X', s=200, edgecolors='black', linewidths=2)
    
    ax.set_title(f'Iteration {idx}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observation:** Watch how the clusters gradually stabilize. Early iterations show large changes, later ones show refinement. This is the iterative optimization in action!

## 10. Measuring Quality: Inertia

### How Good is Our Clustering?

**Inertia** (also called within-cluster sum of squares) measures how tight the clusters are:

$$\text{Inertia} = \sum_{i=1}^{n} \|\mathbf{x}_i - \mathbf{\mu}_{c_i}\|^2$$

Lower inertia = tighter clusters = better fit (for the given $k$).

In [ ]:
def compute_inertia(X: np.ndarray, centroids: np.ndarray, labels: np.ndarray) -> float:
    """
    Compute within-cluster sum of squares (inertia).
    
    Args:
        X: Data points
        centroids: Cluster centers
        labels: Cluster assignments
    
    Returns:
        Inertia value
    """
    inertia = 0
    for i in range(len(centroids)):
        cluster_points = X[labels == i]
        inertia += np.sum((cluster_points - centroids[i]) ** 2)
    return inertia

# Compute inertia for our clustering
inertia = compute_inertia(X, final_centroids, final_labels)
print(f"Final inertia: {inertia:.2f}")

Let's track how inertia decreases over iterations.

In [ ]:
# Compute inertia at each iteration
inertias = []
for centroids, labels in history:
    inertia = compute_inertia(X, centroids, labels)
    inertias.append(inertia)

plt.figure(figsize=(10, 5))
plt.plot(range(len(inertias)), inertias, marker='o', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Inertia')
plt.title('Inertia Decreases as Clusters Improve')
plt.grid(True, alpha=0.3)
plt.show()

**Key insight:** Inertia always decreases (or stays the same) with each iteration. This proves that k-means is **guaranteed to converge** — it's always making progress toward better clusters!

## 11. The Elbow Method: Choosing K

### How Many Clusters?

How do we choose $k$? The **elbow method** runs k-means for different values of $k$ and plots inertia.

Look for an "elbow" — a point where inertia stops decreasing rapidly. That's often a good choice for $k$.

In [ ]:
# Try different values of k
k_values = range(1, 8)
inertias_for_k = []

for k_val in k_values:
    centroids, labels, _ = kmeans(X, k=k_val, max_iters=100)
    inertia = compute_inertia(X, centroids, labels)
    inertias_for_k.append(inertia)
    print(f"k={k_val}: inertia={inertia:.2f}")

Now let's plot the elbow curve.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(k_values, inertias_for_k, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Elbow at k=3')
plt.legend()
plt.show()

**Key insight:** The curve bends sharply at $k=3$ (the "elbow"). Beyond that, adding more clusters doesn't reduce inertia much. This suggests $k=3$ is the natural number of clusters — which matches our data generation!

## 12. Limitations: When K-Means Struggles

### Non-Spherical Clusters

K-means assumes clusters are roughly spherical (circular). Let's see what happens with non-spherical data.

In [ ]:
# Generate two concentric circles
from numpy import linspace, cos, sin, concatenate

def generate_circles(n_samples: int = 200) -> np.ndarray:
    """Generate two concentric circles."""
    n_samples_inner = n_samples // 2
    n_samples_outer = n_samples - n_samples_inner
    
    # Inner circle
    theta_inner = linspace(0, 2 * np.pi, n_samples_inner)
    inner = np.column_stack([
        cos(theta_inner) + np.random.randn(n_samples_inner) * 0.1,
        sin(theta_inner) + np.random.randn(n_samples_inner) * 0.1
    ])
    
    # Outer circle
    theta_outer = linspace(0, 2 * np.pi, n_samples_outer)
    outer = np.column_stack([
        3 * cos(theta_outer) + np.random.randn(n_samples_outer) * 0.1,
        3 * sin(theta_outer) + np.random.randn(n_samples_outer) * 0.1
    ])
    
    return concatenate([inner, outer])

X_circles = generate_circles()

plt.figure(figsize=(8, 6))
plt.scatter(X_circles[:, 0], X_circles[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Concentric Circles - Non-Spherical Clusters')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

Now let's try k-means on this circular data.

In [ ]:
# Run k-means with k=2 (should give inner and outer circle)
centroids_circles, labels_circles, _ = kmeans(X_circles, k=2)

plt.figure(figsize=(8, 6))
for i in range(2):
    cluster_points = X_circles[labels_circles == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
               alpha=0.6, s=50, c=colors[i], label=f'Cluster {i}')

plt.scatter(centroids_circles[:, 0], centroids_circles[:, 1],
           c='red', marker='X', s=300, edgecolors='black', linewidths=2)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('K-Means Struggles with Non-Spherical Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

**Key limitation:** K-means failed! It split the data along a diameter instead of separating inner from outer circle. This is because k-means uses **Euclidean distance** and assumes **convex, spherical clusters**.

For complex shapes, consider algorithms like DBSCAN or spectral clustering.

## 13. The Initialization Problem

### Why Multiple Runs Matter

K-means can converge to different solutions depending on initialization. Let's run it multiple times and see the variance.

In [ ]:
# Run k-means 5 times with different random initializations
n_runs = 5
results = []

for run in range(n_runs):
    centroids, labels, _ = kmeans(X, k=3)
    inertia = compute_inertia(X, centroids, labels)
    results.append((centroids, labels, inertia))
    print(f"Run {run + 1}: inertia = {inertia:.2f}")

# Find best result
best_idx = np.argmin([r[2] for r in results])
worst_idx = np.argmax([r[2] for r in results])
print(f"\nBest run: {best_idx + 1} (inertia = {results[best_idx][2]:.2f})")
print(f"Worst run: {worst_idx + 1} (inertia = {results[worst_idx][2]:.2f})")

Let's compare the best and worst results visually.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for idx, (ax, result_idx, title) in enumerate([
    (axes[0], best_idx, f'Best Result (Inertia: {results[best_idx][2]:.2f})'),
    (axes[1], worst_idx, f'Worst Result (Inertia: {results[worst_idx][2]:.2f})')
]):
    centroids, labels, _ = results[result_idx]
    
    for i in range(k):
        cluster_points = X[labels == i]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1],
                  alpha=0.6, s=50, c=colors[i])
    
    ax.scatter(centroids[:, 0], centroids[:, 1],
              c='red', marker='X', s=300, edgecolors='black', linewidths=2)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight:** Different initializations can lead to different solutions. In practice, run k-means multiple times and pick the best result (lowest inertia). This is called **k-means++** initialization addresses this by choosing initial centroids more intelligently.

## 14. K-Means++ Initialization

### Smarter Starting Points

**K-means++** is a smarter initialization method that spreads initial centroids far apart:

1. Choose first centroid randomly from data
2. For each remaining centroid:
   - Compute distance from each point to nearest existing centroid
   - Choose next centroid with probability proportional to squared distance
3. Run standard k-means

This tends to find better starting positions.

In [ ]:
def initialize_centroids_plus_plus(X: np.ndarray, k: int) -> np.ndarray:
    """
    K-means++ initialization: spread initial centroids far apart.
    
    Args:
        X: Data points
        k: Number of clusters
    
    Returns:
        Initial centroids
    """
    n_samples = X.shape[0]
    centroids = []
    
    # Choose first centroid randomly
    first_idx = np.random.randint(n_samples)
    centroids.append(X[first_idx])
    
    # Choose remaining centroids
    for _ in range(k - 1):
        # Compute distance to nearest centroid for each point
        distances = np.array([min([np.sum((x - c) ** 2) for c in centroids]) for x in X])
        
        # Choose next centroid with probability proportional to distance squared
        probabilities = distances / distances.sum()
        next_idx = np.random.choice(n_samples, p=probabilities)
        centroids.append(X[next_idx])
    
    return np.array(centroids)

# Test k-means++
centroids_pp = initialize_centroids_plus_plus(X, k=3)
print("K-means++ initial centroids:")
print(centroids_pp)

Let's compare random vs k-means++ initialization visually.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Random initialization
centroids_random = initialize_centroids(X, k=3)
axes[0].scatter(X[:, 0], X[:, 1], alpha=0.4, s=50)
axes[0].scatter(centroids_random[:, 0], centroids_random[:, 1],
               c='red', marker='X', s=300, edgecolors='black', linewidths=2)
axes[0].set_title('Random Initialization')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# K-means++ initialization
axes[1].scatter(X[:, 0], X[:, 1], alpha=0.4, s=50)
axes[1].scatter(centroids_pp[:, 0], centroids_pp[:, 1],
               c='red', marker='X', s=300, edgecolors='black', linewidths=2)
axes[1].set_title('K-means++ Initialization')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight:** K-means++ tends to spread centroids across different regions, while random initialization might cluster them together. This leads to faster convergence and better results.

## 15. Practical Considerations

### Summary of Best Practices

**When to use k-means:**
- Clusters are roughly spherical and similar in size
- You know (or can estimate) the number of clusters
- You need a fast, scalable algorithm
- Euclidean distance is meaningful for your data

**How to use it well:**
1. **Normalize features** if they have different scales
2. **Try multiple values of k** and use the elbow method
3. **Run multiple times** with different initializations (or use k-means++)
4. **Evaluate results** with domain knowledge, not just inertia

**When to avoid k-means:**
- Clusters have complex, non-convex shapes
- Clusters have very different sizes or densities
- You have many outliers (k-means is sensitive to them)
- Features are categorical (Euclidean distance doesn't make sense)

## 16. Key Takeaways

### What We Learned

**Core Algorithm:**
- K-means alternates between **assignment** (points to clusters) and **update** (centroids to cluster means)
- It's **guaranteed to converge** because inertia decreases each iteration
- Convergence happens when centroids stop moving significantly

**Key Concepts:**
- **Distance matters:** K-means assumes Euclidean distance and spherical clusters
- **Initialization matters:** Different starting points lead to different solutions
- **K matters:** Use the elbow method to choose the number of clusters
- **Inertia measures fit:** Lower is better, but it always decreases with more clusters

**Limitations:**
- Struggles with non-spherical or overlapping clusters
- Sensitive to outliers and initialization
- Requires specifying k in advance

**Intuition:**
K-means is like a game where:
- **Assignment** says: "Given these centroids, which one is each point closest to?"
- **Update** says: "Given these assignments, where should the centroids be?"
- We alternate until nothing changes — that's **equilibrium**!

This simple idea powers one of the most widely-used clustering algorithms in machine learning.